# Nonword ill-formedness: orthographic neighbourhood surprisal

## Definition

The lexicon is every CMUdict word $w$, weighted by its SUBTLEX-US frequency $f(w) = \text{count}(w) + 1$, so
$P_{\text{lex}}(w) = f(w) / \sum_v f(v)$.

For a nonword $x$, $D(x, w)$ is the letter edit distance (Levenshtein), and $d$ is that distance normalised by the
longer of the two lengths:
$$d(x, w) = \frac{D(x, w)}{\max(|x|, |w|)}$$
The word $x$ itself ($D = 0$) is left out, so real words can be scored the same way.

Neighbourhood surprisal (nats), where higher means farther from every frequent word, i.e. less word-like:
$$S_\beta(x) = -\ln \sum_{w \ne x} P_{\text{lex}}(w)\, e^{-\beta\, d(x, w)}$$

Neighbourhood entropy, where low means one neighbour dominates and high means many are about equally close:
$$P_\beta(w \mid x) = \frac{f(w)\, e^{-\beta d(x, w)}}{\sum_{v \ne x} f(v)\, e^{-\beta d(x, v)}}, \qquad
H_\beta(x) = -\sum_{w \ne x} P_\beta(w \mid x) \ln P_\beta(w \mid x)$$

OLD20 is the mean raw distance $D$ to the 20 nearest words (Yarkoni et al. 2008).

List score: the mean over the list's nonwords. Here $\beta = 20$; the pipeline also writes $\beta = 10$.
Without the normalisation by length, frequent short words (*a*, *to*) dominate the sum, and *xqzt* comes out more
word-like than *blick*.

## Setup (once)

```bash
pip install -e ".[notebooks]"
python -c "import nltk; nltk.download('cmudict')"
linguistic fetch subtlex_us
```

## Input

Either point `STIMULI` to a CSV with columns `stimulus_id,condition,text` (`condition` = `nonword`, with the
nonwords of a list separated by spaces), or type lists into `NONWORD_LISTS` (this overrides `STIMULI`).
`REFERENCE` is a sentence list whose real words are scored the same way, for comparison.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

STIMULI = ROOT / "data/examples/nonwords/stimuli.csv"
NONWORD_LISTS = []      # e.g. ["blick bnick brane", "xqzt zzkpq"]
REFERENCE = ROOT / "data/examples/sentences/stimuli.csv"
BETA = 20
OUT = ROOT / "out" / "nonword_neighbourhood"

OUT.mkdir(parents=True, exist_ok=True)
if NONWORD_LISTS:
    STIMULI = OUT / "stimuli.csv"
    pd.DataFrame({"stimulus_id": [f"list_{i + 1}" for i in range(len(NONWORD_LISTS))],
                  "condition": "nonword", "text": NONWORD_LISTS}).to_csv(STIMULI, index=False)
pd.read_csv(STIMULI).head()

,stimulus_id,condition,text
0,nonword_list_1,nonword,arfase treitege ligose brooslude bermose flish...
1,nonword_list_2,nonword,ifenep fierdane filtose flishtmose hollept fic...
2,nonword_list_3,nonword,guebal heafnure hilenept aisnule mouferund mol...
3,nonword_list_4,nonword,wheilhoke lewnept juilehoke santlude treamurem...
4,nonword_list_5,nonword,pomaph tunfelmose walsund worltnule wisdane ju...


## Step 1: run the pipeline

In [2]:
from linguistic.assemble import run_metrics
from linguistic.io import load_stimuli
from linguistic.normalize import Lexicon, build_units
from linguistic.registry import Context, select

lexicon = Lexicon()


def pipeline(path):
    stimuli = load_stimuli(path)
    units = build_units(stimuli, lexicon)
    return run_metrics(Context(stimuli, units.words, units.phones, lexicon),
                       select(["orthographic_neighbourhood"]))


nw, ref = pipeline(STIMULI), pipeline(REFERENCE)
SCORES = ["orth_old20", f"orth_nbhd_surprisal_b{BETA}", f"orth_nbhd_entropy_b{BETA}"]
nw.words[["stimulus_id", "word", *SCORES]].head()

,stimulus_id,word,orth_old20,orth_nbhd_surprisal_b20,orth_nbhd_entropy_b20
0,nonword_list_1,arfase,2.00,14.216950,4.092841
1,nonword_list_1,treitege,3.00,14.983683,5.741668
2,nonword_list_1,ligose,2.10,13.822490,3.594570
3,nonword_list_1,brooslude,3.90,15.904056,6.035122
4,nonword_list_1,bermose,2.55,14.073299,3.743767


## Step 2: recompute from scratch

The same formulas with CMUdict, the SUBTLEX-US file and `rapidfuzz` alone, checked against step 1.

In [3]:
import re

import nltk
from rapidfuzz.distance import Levenshtein
from rapidfuzz.process import cdist

from linguistic import resources

# lexicon: CMUdict spellings made of a-z only, f = SUBTLEX-US count + 1
subtlex = pd.read_csv(resources.path("subtlex_us"), sep="\t", keep_default_na=False, na_values=[""])
count = dict(zip(subtlex["Word"].astype(str).str.lower(), subtlex["FREQcount"]))
lex = sorted(w for w in {w.lower() for w in nltk.corpus.cmudict.dict()} if re.fullmatch("[a-z]+", w))
f = np.array([count.get(w, 0) + 1 for w in lex], dtype=float)
p_lex = f / f.sum()
lex_len = np.array([len(w) for w in lex])


def neighbourhood(x, beta=BETA):
    x = re.sub("[^a-z]", "", x.lower())
    D = cdist([x], lex, scorer=Levenshtein.distance, workers=-1)[0].astype(float)
    keep = D > 0                                           # leave the word itself out
    D, d = D[keep], D[keep] / np.maximum(len(x), lex_len[keep])
    surprisal = -np.log((p_lex[keep] * np.exp(-beta * d)).sum())
    w = f[keep] * np.exp(-beta * d)
    p = w / w.sum()
    entropy = -(p[p > 0] * np.log(p[p > 0])).sum()
    return {"orth_old20": np.sort(D)[:20].mean(),
            f"orth_nbhd_surprisal_b{beta}": surprisal, f"orth_nbhd_entropy_b{beta}": entropy}


scratch = pd.DataFrame([{"stimulus_id": r.stimulus_id, "word": r.word, **neighbourhood(r.word)}
                        for r in nw.words.itertuples()])
for c in SCORES:
    print(f"{c:24s} max |scratch - pipeline| = {np.abs(scratch[c] - nw.words[c]).max():.2e}")

orth_old20               max |scratch - pipeline| = 0.00e+00
orth_nbhd_surprisal_b20  max |scratch - pipeline| = 1.78e-15
orth_nbhd_entropy_b20    max |scratch - pipeline| = 2.66e-15


## Per nonword

In [4]:
scratch.sort_values(f"orth_nbhd_surprisal_b{BETA}").round(2)

,stimulus_id,word,orth_old20,orth_nbhd_surprisal_b20,orth_nbhd_entropy_b20
21,nonword_list_2,honude,1.95,13.44,3.02
56,nonword_list_5,toithel,2.75,13.52,3.45
2,nonword_list_1,ligose,2.10,13.82,3.59
55,nonword_list_5,werdane,2.55,14.04,3.89
4,nonword_list_1,bermose,2.55,14.07,3.74
59,nonword_list_5,treinege,2.80,14.13,4.08
0,nonword_list_1,arfase,2.00,14.22,4.09
50,nonword_list_5,walsund,2.80,14.29,3.43
10,nonword_list_1,bulnete,2.65,14.34,3.68
25,nonword_list_3,heafnure,3.25,14.36,3.73


## List level: mean over the list's nonwords

In [5]:
lists = scratch.groupby("stimulus_id")[SCORES].mean()
lists.insert(0, "n_nonwords", scratch.groupby("stimulus_id").size())
lists.round(2)

,n_nonwords,orth_old20,orth_nbhd_surprisal_b20,orth_nbhd_entropy_b20
stimulus_id,,,,
nonword_list_1,12,2.98,14.81,4.78
nonword_list_2,12,3.18,15.00,4.90
nonword_list_3,12,3.24,15.08,4.98
nonword_list_4,12,3.72,15.41,5.65
nonword_list_5,12,3.15,14.90,4.76


## Examples: lists with the highest and lowest mean surprisal

In [6]:
col = f"orth_nbhd_surprisal_b{BETA}"
text = pd.read_csv(STIMULI).set_index("stimulus_id")["text"]
ranked = lists.sort_values(col, ascending=False)


def show(ids, label):
    print(label)
    for sid in ids:
        print(f"  {ranked.loc[sid, col]:.2f} nats  {sid}: {text[sid]}")


show(ranked.index[:2], "High surprisal (least word-like)")
show(ranked.index[::-1][:2], "Low surprisal (most word-like)")

High surprisal (least word-like)
  15.41 nats  nonword_list_4: wheilhoke lewnept juilehoke santlude treamuremose selserane trocklude stanelabe frintase dautege carfetrel teblenept
  15.08 nats  nonword_list_3: guebal heafnure hilenept aisnule mouferund molkrane moothoke nishtlude nistnane parmure horpenure poiltvose
Low surprisal (most word-like)
  14.81 nats  nonword_list_1: arfase treitege ligose brooslude bermose flishnule blouduck lotternule brealrel breistel bulnete roifel
  14.90 nats  nonword_list_5: pomaph tunfelmose walsund worltnule wisdane jubelabe wolprel werdane toithel tivenul tertlure treinege


## Compared with real words

In [7]:
both = pd.concat([nw.words.assign(kind="nonword"), ref.words.assign(kind="real word")])
both["n_letters"] = both["word"].str.lower().str.count("[a-z]")
both.groupby("kind")[[*SCORES, "n_letters"]].agg(["mean", "std"]).round(2)

orth_old20       orth_nbhd_surprisal_b20        \
                mean   std                    mean   std   
kind                                                       
nonword         3.25  0.79                   15.04  0.66   
real word       1.24  0.40                   11.78  1.44   

          orth_nbhd_entropy_b20       n_letters        
                           mean   std      mean   std  
kind                                                   
nonword                    5.01  0.82      7.97  1.23  
real word                  2.26  0.83      3.93  1.58

In [8]:
# how the scores relate to each other and to length (Spearman, nonwords)
both.loc[both["kind"] == "nonword", [*SCORES, "n_letters"]].corr(method="spearman").round(2)

,orth_old20,orth_nbhd_surprisal_b20,orth_nbhd_entropy_b20,n_letters
orth_old20,1.00,0.76,0.54,0.91
orth_nbhd_surprisal_b20,0.76,1.00,0.72,0.65
orth_nbhd_entropy_b20,0.54,0.72,1.00,0.58
n_letters,0.91,0.65,0.58,1.00


All three scores grow with length: longer strings have fewer close neighbours.

The phone version of the same measures is `phonological_neighbourhood`. It needs phones, and none of these nonwords
is in CMUdict, so it is empty until hand phone labels are given
(see `data/examples/nonwords/output/missing_phones.csv`).